In [44]:
import pandas as pd
import numpy as np
from common.county_geometry import convert_points
from common.county_geometry import central_angle, earth_radius_mi

data_path = "./data/cleaner/intersections_clean.json"

def read_in_intersections(path_to_intersection_data):
    df = pd.read_json(path_to_intersection_data, orient='records', lines=True)
    df['GEOMETRY'] = df.GEOMETRY.apply(np.array)
    df['KY_grid_XY'] = df.KY_grid_XY.apply(np.array)
    return df.set_index("INTID")

intersections = read_in_intersections(data_path)
intersections.head()

,FST_ROADNAME,SEC_ROADNAME,SIFCODE1,SIFCODE2,FST_SIFID,SEC_SIFID,GEOMETRY,KY_grid_XY
INTID,,,,,,,,
5710837346,REHL RD,W REHL CT,5464,7662,4976,6856,"[-85.51044384084946, 38.20588660809318]","[1278243.0, 259531.9375]"
10005800273,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52816882852979, 38.20037561255241]","[1273126.375, 257588.25]"
14300767569,REHL RD,TUCKER STATION RD,5464,6551,4976,5908,"[-85.52841872335158, 38.200360349057064]","[1273050.50875001, 257590.85375001]"
18011691414,I 64 EAST,I 265 RAMP,3194,9996,3076,8763,"[-85.50495414554669, 38.22260344055154]","[1279903.25, 265597.5]"
23945910678,I 265 NORTH,I 265 RAMP,9349,9996,8197,8763,"[-85.50554642815675, 38.222127288727606]","[1279730.75, 265426.4375]"


##### Info about X_COORD YCOORD system

via: https://www.lojic.org/data/projection-information

For this system, the Commonwealth shall be divided into a north zone and a south zone. The north zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 37 degrees, 58 minutes, and 38 degrees, 58 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 84 degrees, 15 minutes west of Greenwich, and the parallel 37 degrees, 30 minutes north latitude. This origin shall be given the coordinates: N=0, E=500,000.000 meters. The south zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 36 degrees, 44 minutes, and 37 degrees, 56 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 85 degrees, 45 minutes west of Greenwich, and the parallel 36 degrees, 20 minutes north latitude. This origin shall be given the coordinates: N=500,000.000, E=500,000.000 meters. The southern edge of the following counties shall delineate the boundary between the north zone and the south zone: Bullitt, Spencer, Anderson, Woodford, Jessamine, Fayette, Clark, Montgomery, Menifee, Morgan, and Lawrence.

One U. S. survey foot equals (1200)/(3937) meter. For conversion of meters to U. S. survey feet, multiply the meters by 3.28083333333 to twelve (12) significant figures. When converting from meters to feet, the conversion factor defined by the U. S. survey foot shall be used.


The plane coordinate values for a point on the earth's surface, used to express the geographic position or location of the point in the appropriate zone of this system, shall consist of two (2) distances expressed in U. S. survey feet and decimals of a foot when using the Kentucky Coordinate System of 1983. For the Kentucky Coordinate System of 1983, one (1) of the distances, to be known as the "northing" or "N", shall give the position in a north/south direction. The other, to be known as the "easting" or "E" shall give the position in an east/west direction. These coordinates shall be made to depend upon and conform to plane rectangular coordinates values for the monumented points of the North American National Geodetic Horizontal Network as published by the National Ocean Service/National Geodetic Survey, and whose plane coordinates have been computed on the systems established by the National Ocean Service/National Geodetic Survey. Any such station may be used for establishing a survey connection to the Kentucky Coordinate System of 1983.

In [45]:
diff = (intersections.GEOMETRY.apply(convert_points.point_to_grid) - intersections.KY_grid_XY)

# Difference between converted GEOMETRY and KY_grid
# KY_grid coordinates are given as distances in feet, so subtrcting one from the other also gives 
# the difference in feet, which can easily be converted to miles. 

grid_distance = diff.apply(np.linalg.norm) # norm((a, b)) -> sqrt(a**2 + b**2)
grid_distance

INTID
5710837346          2.994226
10005800273        11.190484
14300767569         2.994305
18011691414         2.995155
23945910678         2.995139
                     ...    
847259462915456     2.998237
847265632743817     2.994492
847261337735316     2.994492
847274257577475     2.992074
847272347859222     2.991810
Length: 20946, dtype: float64

In [46]:
LL_angle = (intersections.KY_grid_XY.apply(convert_points.point_to_ll).combine(
    intersections.GEOMETRY, central_angle))

# difference between GEOMETRY and converted KY_grid in US feet as determined by haversine algorithm

hav_distance = LL_angle*earth_radius_mi*5280 # convert angle to distance in US feet
hav_distance

INTID
5710837346          2.994831
10005800273        11.191567
14300767569         2.994905
18011691414         2.995760
23945910678         2.995744
                     ...    
847259462915456     2.998832
847265632743817     2.995095
847261337735316     2.995095
847274257577475     2.992651
847272347859222     2.992373
Length: 20946, dtype: float64

In [47]:
(hav_distance-grid_distance).abs().describe()


count    20946.000000
mean         0.002402
std          0.008413
min          0.000005
25%          0.000535
50%          0.000565
75%          0.000590
max          0.398428
dtype: float64

In [48]:

# biggest difference is 0.398428
0.398428 * 12 # convert to inches -> 4.781136 pretty close
# average distance
0.002402 * 12# -> 0.028824000000000002 Just under 1/32 inch. VERY close.


0.028824000000000002

#### Conclusion

Very little difference between the projections when they're converted to be compatible.